# Session 2 — Triggering the Retrain Pipeline

**Goal:** Trigger the retrain job the same way a CI/CD system would, then understand how
the deployment gate protects production from worse models.

The instructor has pre-deployed the `workshop_retrain_churn` job using:
```bash
databricks bundle deploy --target dev
```

You'll trigger a run via the Databricks SDK. This time the job trains on **`churn_training_data`** —
the combined original + drifted dataset you created in `02_simulate_drift.ipynb` — so the new
model actually learns from the distribution shift that triggered the alert.

In [0]:
%run ../utils/config

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Find the retrain job by name
jobs_list = list(w.jobs.list(name=f"[dev {safe_username}] workshop_retrain_churn"))
if not jobs_list:
    raise ValueError(f"Job '[dev {schema}] workshop_retrain_churn' not found. Has the instructor deployed the bundle?")

job = jobs_list[0]
print(f"Found job: {job.settings.name}")
print(f"Job ID   : {job.job_id}")

In [0]:
# Trigger a run — training on the combined original + drifted dataset
run_response = w.jobs.run_now(
    job_id=job.job_id,
    job_parameters={
        "source_table": f"{catalog}.{schema}.telco_churn_new_training_data",
    },
)
run_id = run_response.run_id
print(f"Run triggered!")
print(f"Run ID  : {run_id}")
print(f"Source  : {catalog}.{schema}.telco_churn_new_training_data")
print(f"View at : {w.config.host}#job/{job.job_id}/run/{run_id}")

## Navigate to the Job Run

1. Click the URL printed above (or go to **Workflows → Jobs** in the left sidebar)
2. Find `workshop_retrain_churn` and click the running run
3. Observe:
   - The **task dependency graph** — tasks turn green as they complete
   - Click any task to see its **log output**
   - The training task will log a link to the MLflow experiment

> The full run takes **4-8 minutes**.

## The Job Runs Asynchronously

The retrain pipeline runs 4 tasks in sequence:

```
Task 1: ingest_and_validate   → fails if data quality checks fail
Task 2: train_models          → logs 3 MLflow runs, picks best F1
Task 3: gate_and_register     → fails if new model is worse than champion
Task 4: update_endpoint       → updates the serving endpoint to the new champion
```

Watch progress in the Workflows UI via the URL printed above. The full pipeline takes **4-8 minutes**.

> You don't need to wait here — move on to the next section while it runs in the background.

## How the Deployment Gate Works

Task 3 (`gate_and_register`) is the quality gate. It prevents a newly trained model from
replacing the champion if the new model is actually worse.

**Gate logic:**
- Load the **challenger** (best run from the current training job)
- Load the **champion** (current `@champion` alias in Unity Catalog)
- Compare F1 scores on the held-out test set
- **Pass:** challenger F1 ≥ champion F1 − 0.05 → register challenger, promote `@champion` alias
- **Fail:** F1 drops more than 5% → raise an exception, Task 4 never runs, endpoint unchanged

The code below runs automatically inside the pipeline. It is shown for reference only — you do not need to run it here.

In [0]:
# ── REFERENCE ONLY: This code runs automatically in Task 3 of the retrain pipeline ──
# Do not run this cell during the workshop.

# from churn_model.evaluate import get_best_run, evaluate_gate
# import yaml
#
# with open("../common/config.yml") as f:
#     config = yaml.safe_load(f)
#
# # Find the challenger (best MLflow run from the current training job)
# challenger_run = get_best_run(experiment_name=config["experiment_name"])
#
# # Compare challenger vs current @champion — raises ValueError if gate fails
# decision = evaluate_gate(
#     challenger_run_id=challenger_run.info.run_id,
#     model_name=f"{catalog}.{schema}.churn_classifier",
#     threshold=0.05,       # Max allowable F1 drop vs champion
# )
#
# print(f"Gate result   : {decision['decision']}")
# print(f"Champion F1   : {decision['champion_f1']:.4f}")
# print(f"Challenger F1 : {decision['challenger_f1']:.4f}")
# print(f"Delta         : {decision['delta']:+.4f}")

## Discussion

- Which task in the pipeline do you expect to take the longest? Why?
- What happens to the endpoint if the gate fails? (Hint: Task 4 never runs)
- When would you tighten the gate threshold below 5%? When would you loosen it?
- Why did we train on the **combined** (original + drifted) dataset rather than just the drifted data?
  What risk would "replace" carry vs. "append"?

➡️ Next: `06_ab_canary_setup.ipynb` — safe rollout with canary traffic splitting